<a href="https://colab.research.google.com/github/BhavanaVuyyuru/pyspark-data-processing/blob/main/notebooks/pyspark_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install PySpark
!pip install pyspark==3.5.0 -q
print("✅ PySpark installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
✅ PySpark installed!


In [2]:
# Install PySpark
!pip install pyspark==4.0.0 -q
print("✅ PySpark installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 12.0 MB/s eta 0:00:00
✅ PySpark installed!


In [3]:
import os
os.makedirs("data", exist_ok=True)

data = """order_id,customer_id,product,quantity,amount,order_date
1001,C001,Laptop,1,999.99,2024-01-05
1002,C002,Phone,2,599.99,2024-01-06
1003,C001,Headphones,1,149.99,2024-01-07
1004,C003,Tablet,1,449.99,2024-01-08
1005,C002,Laptop,1,999.99,2024-01-09
1006,C004,Phone,3,599.99,2024-01-10
1007,C001,Charger,2,29.99,2024-01-11
1008,C003,Laptop,1,999.99,2024-01-12
1009,C005,Headphones,2,149.99,2024-01-13
1010,C004,Tablet,1,449.99,2024-01-14"""

with open("data/orders.csv", "w") as f:
    f.write(data)

print("✅ Sample data created!")

✅ Sample data created!


In [4]:
from pyspark.sql import SparkSession

# Start Spark
spark = SparkSession.builder \
    .appName("OrderProcessing") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark session started!")

# Load data
df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
print(f"📦 Total records loaded: {df.count()}")
print("\n📋 Schema:")
df.printSchema()
print("\n📋 Sample data:")
df.show(5)

✅ Spark session started!
📦 Total records loaded: 10

📋 Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)


📋 Sample data:
+--------+-----------+----------+--------+------+----------+
|order_id|customer_id|   product|quantity|amount|order_date|
+--------+-----------+----------+--------+------+----------+
|    1001|       C001|    Laptop|       1|999.99|2024-01-05|
|    1002|       C002|     Phone|       2|599.99|2024-01-06|
|    1003|       C001|Headphones|       1|149.99|2024-01-07|
|    1004|       C003|    Tablet|       1|449.99|2024-01-08|
|    1005|       C002|    Laptop|       1|999.99|2024-01-09|
+--------+-----------+----------+--------+------+----------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import col

# Clean
df_clean = df \
    .dropna(subset=["order_id", "amount", "customer_id"]) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .filter(col("amount") > 0)

print(f"🧹 Clean records: {df_clean.count()}")
df_clean.show()

🧹 Clean records: 10
+--------+-----------+----------+--------+------+----------+
|order_id|customer_id|   product|quantity|amount|order_date|
+--------+-----------+----------+--------+------+----------+
|    1001|       C001|    Laptop|       1|999.99|2024-01-05|
|    1002|       C002|     Phone|       2|599.99|2024-01-06|
|    1003|       C001|Headphones|       1|149.99|2024-01-07|
|    1004|       C003|    Tablet|       1|449.99|2024-01-08|
|    1005|       C002|    Laptop|       1|999.99|2024-01-09|
|    1006|       C004|     Phone|       3|599.99|2024-01-10|
|    1007|       C001|   Charger|       2| 29.99|2024-01-11|
|    1008|       C003|    Laptop|       1|999.99|2024-01-12|
|    1009|       C005|Headphones|       2|149.99|2024-01-13|
|    1010|       C004|    Tablet|       1|449.99|2024-01-14|
+--------+-----------+----------+--------+------+----------+



In [6]:
from pyspark.sql.functions import sum as _sum, avg, count, round as _round, min as _min, max as _max

# Aggregate by customer
customer_stats = df_clean.groupBy("customer_id").agg(
    _round(_sum("amount"), 2).alias("total_spend"),
    _round(avg("amount"), 2).alias("avg_order_value"),
    count("order_id").alias("total_orders"),
    _sum("quantity").alias("total_items"),
    _min("order_date").alias("first_order"),
    _max("order_date").alias("last_order")
)

print("📊 Customer Summary:")
customer_stats.orderBy(col("total_spend").desc()).show()

📊 Customer Summary:
+-----------+-----------+---------------+------------+-----------+-----------+----------+
|customer_id|total_spend|avg_order_value|total_orders|total_items|first_order|last_order|
+-----------+-----------+---------------+------------+-----------+-----------+----------+
|       C002|    1599.98|         799.99|           2|          3| 2024-01-06|2024-01-09|
|       C003|    1449.98|         724.99|           2|          2| 2024-01-08|2024-01-12|
|       C001|    1179.97|         393.32|           3|          4| 2024-01-05|2024-01-11|
|       C004|    1049.98|         524.99|           2|          4| 2024-01-10|2024-01-14|
|       C005|     149.99|         149.99|           1|          2| 2024-01-13|2024-01-13|
+-----------+-----------+---------------+------------+-----------+-----------+----------+



In [7]:
from pyspark.sql.functions import rank
from pyspark.sql.window import Window

# Rank customers by total spend
window_spec = Window.orderBy(col("total_spend").desc())

customer_ranked = customer_stats.withColumn(
    "spend_rank", rank().over(window_spec)
)

print("🏆 Customer Spend Ranking:")
customer_ranked.orderBy("spend_rank").show()

🏆 Customer Spend Ranking:
+-----------+-----------+---------------+------------+-----------+-----------+----------+----------+
|customer_id|total_spend|avg_order_value|total_orders|total_items|first_order|last_order|spend_rank|
+-----------+-----------+---------------+------------+-----------+-----------+----------+----------+
|       C002|    1599.98|         799.99|           2|          3| 2024-01-06|2024-01-09|         1|
|       C003|    1449.98|         724.99|           2|          2| 2024-01-08|2024-01-12|         2|
|       C001|    1179.97|         393.32|           3|          4| 2024-01-05|2024-01-11|         3|
|       C004|    1049.98|         524.99|           2|          4| 2024-01-10|2024-01-14|         4|
|       C005|     149.99|         149.99|           1|          2| 2024-01-13|2024-01-13|         5|
+-----------+-----------+---------------+------------+-----------+-----------+----------+----------+



In [8]:
# Aggregate by product
product_stats = df_clean.groupBy("product").agg(
    count("order_id").alias("times_ordered"),
    _sum("quantity").alias("total_units_sold"),
    _round(_sum("amount"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc())

print("🛒 Product Revenue Summary:")
product_stats.show()

🛒 Product Revenue Summary:
+----------+-------------+----------------+-------------+
|   product|times_ordered|total_units_sold|total_revenue|
+----------+-------------+----------------+-------------+
|    Laptop|            3|               3|      2999.97|
|     Phone|            2|               5|      1199.98|
|    Tablet|            2|               2|       899.98|
|Headphones|            2|               3|       299.98|
|   Charger|            1|               2|        29.99|
+----------+-------------+----------------+-------------+



In [9]:
os.makedirs("output", exist_ok=True)

# Save results
customer_ranked.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/customer_stats")

product_stats.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/product_stats")

print("✅ Results saved!")
print("\n🎉 PySpark Pipeline Complete!")

spark.stop()
print("✅ Spark session stopped")

✅ Results saved!

🎉 PySpark Pipeline Complete!
✅ Spark session stopped
